# Prediction and suppression neurons

How many MLP neurons have output directions strongly aligned or anti-aligned
with a small subset of the vocabulary? Classify neurons from checkpoint weights
and plot their fractions across physical depth for feedforward and recurrent models.
No inputs, activation collection, or forward passes are needed.

Both modes analyze the same checkpoints and produce the same two figures.
The recurrent panels follow the shared `all` paper group; feedforward panels add
the matched Parcae controls to GPT-2, Llama, TinyLlama, and OLMo-2.
`mode="quick"` saves only in a new run directory; `mode="full"` also refreshes the
paper figures. Recurrence counts do not change the weights, so each checkpoint
is loaded once and each physical layer is measured once.

In [ ]:
mode = "quick"  # "full" also updates paper figures.
feedforward_models = None  # Standard feedforward models plus matched Parcae controls.
looped_models = None  # Shared PAPER_MODEL_GROUPS["all"], in panel order.
kurtosis_threshold = 10.0
neuron_chunk_size = 256
device = "auto"
dtype = "auto"  # Checkpoint precision; cosine moments always use float32.
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.
source_run = None  # Optional completed run directory with matching scientific settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from experiments.cache import ExperimentCache
from experiments.paper_models import PAPER_MODEL_GROUPS
from experiments.saved_runs import SavedRun
from experiments.support import (
    BASELINE_FOR,
    MODEL_LABELS,
    export_paper_figures,
    model_subplots,
    save_figure,
    save_results,
)
from experiments.weight_support import (
    collect_weight_summary,
    configure_weight_experiment,
)
from looped_internals.analysis.neurons import (
    classify_neurons,
    neuron_vocab_cosine_moments,
)

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# GPT-2 is specific to this weight comparison; all other labels are shared.
MODEL_LABELS = {"gpt2": "GPT-2"} | MODEL_LABELS
if looped_models is None:
    looped_models = list(PAPER_MODEL_GROUPS["all"])
if feedforward_models is None:
    parcae_controls = dict.fromkeys(
        BASELINE_FOR[name] for name in looped_models if name.startswith("parcae_")
    )
    feedforward_models = ["gpt2", "llama_1b", "tinyllama", "olmo2", *parcae_controls]

# Measure each checkpoint once, preserving panel order.
models = list(dict.fromkeys([*feedforward_models, *looped_models]))

config = configure_weight_experiment(
    models=models,
    mode=mode,
    device=device,
    dtype=dtype,
    output_dir=output_dir,
)
if type(neuron_chunk_size) is not int or neuron_chunk_size < 1:
    raise ValueError("neuron_chunk_size must be a positive integer")
if not np.isfinite(kurtosis_threshold):
    raise ValueError("kurtosis_threshold must be finite")

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
def summarize_neurons(bridge, structure):
    """Classify physical-layer neurons, reducing each vocabulary comparison at once."""
    # The current bridge exposes TL orientations: W_U (D, V), W_out (N, D).
    # Normalize the full vocabulary once, then keep only one neuron chunk's cosines.
    vocabulary = F.normalize(bridge.W_U.float(), dim=0, eps=1e-8)
    summary = {
        "neurons_per_layer": [],
        "prediction_count": [],
        "suppression_count": [],
        "prediction_fraction": [],
        "suppression_fraction": [],
    }
    for group, index in structure.iter_layers():
        weights = getattr(bridge, group.hook_prefix)[index].mlp.W_out
        counts = torch.zeros(2, dtype=torch.int64, device=weights.device)
        for chunk in weights.split(neuron_chunk_size):
            cosine = F.normalize(chunk.float(), dim=-1, eps=1e-8) @ vocabulary
            cosine.clamp_(-1, 1)
            moments = neuron_vocab_cosine_moments(cosine)
            labels = classify_neurons(
                moments["kurtosis"], moments["skew"], kurtosis_threshold
            )
            counts += torch.stack([(labels == 1).sum(), (labels == -1).sum()])
            del cosine, moments, labels
        prediction, suppression = counts.cpu().tolist()
        neurons = len(weights)
        summary["neurons_per_layer"].append(neurons)
        summary["prediction_count"].append(prediction)
        summary["suppression_count"].append(suppression)
        summary["prediction_fraction"].append(prediction / neurons)
        summary["suppression_fraction"].append(suppression / neurons)
    return summary

## Load and reduce weights

The shared retrofit names select checkpoints **trained at 32 recurrences**;
no inference rollout is performed. Parcae controls were trained with a single
core pass. Parcae 770M, 140M, and 370M each use their matched control.
Load each checkpoint, measure its MLP output weights against every unembedding
column, and release it before loading the next.
Cosines and their moments use float32; `dtype` controls checkpoint loading only.

Each neuron chunk is reduced immediately. We keep per-layer counts and fractions,
plus checkpoint revisions, dimensions, and layer groups in `overview.pt`.
The figure cells below reuse these compact summaries without loading models again.

In [ ]:
settings = asdict(config)
settings.update(
    output_dir=str(config.output_dir),
    feedforward_models=feedforward_models,
    looped_models=looped_models,
    kurtosis_threshold=kurtosis_threshold,
    neuron_chunk_size=neuron_chunk_size,
    moment_dtype="float32",
    versions={
        name: version(name) for name in ("torch", "transformer-lens", "transformers")
    },
)
settings["cache"] = result_cache.settings
previous = SavedRun(source_run, settings, analysis_version=analysis_version)
previous.check_pairs(settings)
settings.update(analysis_version=analysis_version, source_run=source_run)
run_dir = save_results({}, settings, config.output_dir)

summaries, run_info = {}, {}
for name in config.models:
    if name in previous:
        summaries[name], run_info[name] = (
            previous.data["summaries"][name],
            previous.info(name),
        )
        continue
    print(f"Analyzing {MODEL_LABELS.get(name, name).replace(chr(10), ' ')}...")
    summaries[name], run_info[name] = collect_weight_summary(
        name,
        config,
        summarize_neurons,
        cache=result_cache,
        analysis_version=analysis_version,
        analysis_options={
            "kurtosis_threshold": kurtosis_threshold,
            "neuron_chunk_size": neuron_chunk_size,
        },
    )

torch.save({"summaries": summaries, "run": run_info}, run_dir / "overview.pt")
result_cache.write_manifest(run_dir)

## What the curves measure

For each neuron, form the distribution of cosine similarities between its MLP
output vector and **all unembedding columns**, including any padded vocabulary
entries. Each column has equal weight. The classification uses population moments
of this distribution: skewness $E[z^3]$ and excess kurtosis $E[z^4]-3$, where
$z=(c-E[c])/\sqrt{\mathrm{Var}(c)}$. Standard deviation and vector norms are floored
at $10^{-8}$; a zero output vector is classified as other.

| Class | Excess kurtosis | Skewness |
| --- | --- | --- |
| Prediction | At least `kurtosis_threshold` | Positive |
| Suppression | At least `kurtosis_threshold` | Zero or negative |
| Other | Below `kurtosis_threshold` | Any |

The default threshold is 10. Each plotted fraction divides the class count by
**all MLP neurons in that layer**, including other neurons. These are weight-based
labels following Gurnee et al. (2024), §5.1, rather than measured effects on model
predictions. We apply no weight centering or normalization-layer folding; MLP gates,
biases, activations, and downstream transformations are not part of this measurement.
Only cosines enter the classification; Euclidean distances are not computed.

Depth runs from the first to last **physical layer** (0–100%), counting recurrent
weights once. Prelude/core/coda boundaries follow the saved layer groups.
There is no averaging across inputs, recurrences, or seeds, and no uncertainty
band: the full set of neurons and vocabulary columns is measured for each checkpoint.
Panel y-axes retain their own scales, so read the tick labels when comparing models.

In [ ]:
def plot_neuron_fractions(names, filename):
    if not names:
        return
    fig, axes = model_subplots(len(names), height_per_subplot=1.8)
    for ax, name in zip(axes, names, strict=True):
        summary = summaries[name]
        n_layers = len(summary["neurons_per_layer"])
        x = np.linspace(0, 100, n_layers)
        for field, label, color in (
            ("prediction_fraction", "Prediction", "#0071bc"),
            ("suppression_fraction", "Suppression", "#d55e00"),
        ):
            ax.plot(x, summary[field], color=color, linewidth=0.8, label=label)
        ax.set(title=MODEL_LABELS.get(name, name), xlabel="Depth (%)", xlim=(0, 100))
        ax.set_ylim(bottom=0)
        if ax.get_subplotspec().is_first_col():
            ax.set_ylabel("Fraction of neurons")

        groups = [
            group for group in run_info[name]["layer_groups"] if group["n_layers"]
        ]
        if len(groups) > 1:
            group_labels = {
                "prelude_blocks": "pre",
                "core_blocks": "core",
                "coda_blocks": "coda",
            }
            start = 0
            scale = 100 / max(n_layers - 1, 1)
            for group in groups:
                if start:
                    ax.axvline(
                        (start - 0.5) * scale,
                        color="#949494",
                        linestyle="--",
                        linewidth=0.5,
                    )
                midpoint = (start + (group["n_layers"] - 1) / 2) * scale
                ax.text(
                    midpoint,
                    0.97,
                    group_labels.get(group["hook_prefix"], group["hook_prefix"]),
                    transform=ax.get_xaxis_transform(),
                    ha="center",
                    va="top",
                    color="#949494",
                )
                start += group["n_layers"]
    fig.legend(
        *axes[0].get_legend_handles_labels(), loc="outside upper center", ncols=2
    )
    save_figure(fig, run_dir, filename, bbox_inches=None)

## Feedforward models

GPT-2, Llama 3.2 1B, TinyLlama, and OLMo-2, followed by the Parcae controls matched
to the recurrent selection (770M, 140M, and 370M). Both figures use the shared
full-width grid, with up to four panels per row and 1.8 inches of height per row.
Blue and orange curves show prediction and suppression fractions, respectively.

In [ ]:
plot_neuron_fractions(feedforward_models, "feedforward")

## Recurrent models

The shared `all` group places the four main-paper models on the first row and the
four appendix models on the second. Parcae 770M is in the main row; 140M and
370M are in the appendix. The marked sections separate prelude, recurrent core,
and coda where present.

In [ ]:
plot_neuron_fractions(looped_models, "looped")

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")